# Coupling JCM and Veros

Couple JCM and Veros using JAX-ESM (JEM).

In [1]:
from pathlib import Path
import jax
#jax.config.update("jax_enable_x64", False) 
import jax.numpy as jnp # for interaction
import numpy as np # to take average of output
import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords
from jcm.terrain import TerrainData
from jcm.forcing import ForcingData
from importlib import resources

import jax_datetime as jdt
import xarray as xr

from jem.components import JCM, Veros, SlabOceanModel
from jem.mapping import IdentityRegridder
from jem.mapping import BasicMapper
from jem.base.coupler import Coupler
import jem.utils.tree_tools as tree_tools

use_ipython = 'get_ipython' in globals()

# Check available devices
print(f"Available devices: {jax.devices()}")
print(f"Number of devices: {len(jax.devices())}")

jax._src.xla_bridge: 2026-03-10 20:58:24,034 WARNING: An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Setting veros.runtime_settings...
Importing core modules
 Using computational backend jax on cpu
  Kernels are compiled during first iteration, be patient
 Runtime settings are now locked

Available devices: [CpuDevice(id=0)]
Number of devices: 1


## Choose terrain
In this example, you can specify one of the three configurations, "aquaplanet", "toy_earth", and "capped_earth", when calling the function `modify_jcm_terrain`. Because Veros cannot simulate poles, this example uses a slab model to cap the poles. We choose slab ocean model to be "fake land". Using slab land model currently will yield unrealistic temperature at poles because the albedo of ice and snow is not implemented in idealize setup.

In [2]:
from modify_jcm_terrain import modify_jcm_terrain
from jem.tool_scripts.generate_jcm_forcing_and_topography_files import generate_jcm_forcing_and_topography_files

truncation_number = 31
total_simulation_time = jdt.to_timedelta(10, "day")

jcm_files = generate_jcm_forcing_and_topography_files(
    resolution=truncation_number,
)
# There are three choices: "aquaplanet", "toy_earth", and "capped_earth". The outcome will be saved in the folder "data"
coords = get_speedy_coords(spectral_truncation=truncation_number)
modified_jcm_terrain_file = modify_jcm_terrain(jcm_files["terrain"], "aquaplanet", "./data")
terrain = TerrainData.from_file(
    modified_jcm_terrain_file,
    coords=coords
)

Using input data directory: "/home/xtt/.cache/jcm".
Check file: /home/xtt/.cache/jcm/terrain_t31.nc... found.
Check file: /home/xtt/.cache/jcm/forcing_t31.nc... found.
Modifying reference jcm terrain file: /home/xtt/.cache/jcm/terrain_t31.nc
Target output file: /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/data/terrain_aquaplanet.nc
Loaded xarray metadata
<xarray.Dataset> Size: 75kB
Dimensions:  (lon: 96, lat: 48)
Coordinates:
  * lon      (lon) float64 768B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
    lev      float64 8B ...
Data variables:
    lsm      (lon, lat) float64 37kB ...
    orog     (lon, lat) float64 37kB ...
Attributes:
    CDI:          Climate Data Interface version 1.7.1 (http://mpimet.mpg.de/...
    CDO:          Climate Data Operators version 1.7.1 (http://mpimet.mpg.de/...
    grid_type:    gaussian
    history:      Sun Apr 21 

## Configurations

In [3]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(24, "hour")
output_dir = (Path(f"output_T{truncation_number}") /"02-02_experimental_JCM_Veros").resolve()

output_dir.mkdir(exist_ok=True, parents=True)
one_second = jdt.to_timedelta(1, "second")

## Create Components

### Create JCM

In [4]:
atm_model = jcm.model.Model(
    coords = coords,
    start_date=start_datetime,
    terrain = terrain,
)

JCM.make_jem_compatible(
    atm_model,
    coupling_timestep=coupling_timestep,
)
    
atm_D2_nodal_shape = atm_model.coords.nodal_shape[1:]

### Create Veros

First need to remove `output_veros.*.nc` files, otherwise veros complains.

In [5]:
import glob
import os
files = glob.glob("output_veros.*.nc")
for f in files:
    print(f"Deleting: {f}")
    os.remove(f)

In [6]:
from veros_case_setup import generateVerosSetup
ocn_model = generateVerosSetup(
    nx = atm_D2_nodal_shape[0],
    ny = atm_D2_nodal_shape[1],
    land_sea_mask_file = modified_jcm_terrain_file,
    dt_mom = 3600.0,
    dt_tracer = 3600.0,
)()
ocn_model.setup()
Veros.make_jem_compatible(
    ocn_model,
    coupling_timestep=coupling_timestep,
)

Veros path: /home/xtt/projects/project_jax-esm/veros_repo/veros-jittable/veros/__init__.py
Running model setup
Initializing streamfunction method
 Solving for boundary contribution by island 0
 Solving for boundary contribution by island 1
Diffusion grid factor delta_iso1 = 0.01894596332563102


<veros_case_setup.generateVerosSetup.<locals>.VerosCaseSetup at 0x746c800dbb60>

### Create Slab Ocean model

In [7]:
slab_ocn_model=SlabOceanModel(
    grid_specification=f"JCM::T{truncation_number:d}",
    start_datetime=start_datetime,
    timestep=coupling_timestep/one_second,
    mask_file=modified_jcm_terrain_file,
    forcing_method=None,
    mask_value=1.0,
)

## Creating Flux and Scalar Exchange between Components

Here we demonstrote the flexibility of JEM: You do not have to use the `BasicMapper` that JEM provides. You can define your own mapping function. In this example, we simply define a function `interaction` as below.

In [8]:
# Creating regridders and mapping
def veros_to_jcm_regridder(arr):
    return arr #jnp.pad(arr, ((0, 0), (4, 4)), constant_values=150)
def jcm_to_veros_regridder(arr):
    return arr#[:, 4:-4]

# Note: Remember to return the `coupled_carry` at the end.
def interaction(coupled_carry):
    atm = coupled_carry["atm"]
    ocn = coupled_carry["ocn"]
    slab_ocn = coupled_carry["slab_ocn"]

    # ===== compute wind stress begin =====
    # Tien-Yiao's ad-hoc way to compute wind stress
    # This conveniently demonstrates how the flux computation can be its own
    # function or module
    drag_coefficient = 1e-3 # dimensionless
    air_density = 1.22 # kg / m^3
    wind_x = jcm_to_veros_regridder(atm["derived"]["physics"].surface_flux.u0)
    wind_y = jcm_to_veros_regridder(atm["derived"]["physics"].surface_flux.v0)
    wind_velocity = jnp.sqrt(wind_x**2 + wind_y**2)    
    vs = ocn["state"].variables
    surface_taux = drag_coefficient * air_density * wind_velocity * wind_x
    surface_tauy = drag_coefficient * air_density * wind_velocity * wind_y
    # ===== compute wind stress end =====

    # This is an ongoing investigation of the instability happens around day 50~60
    # The heat flux becomes unphyiscally large, which eventually (do not know why)
    # crash the coupled simulation. Here I use jnp.clip to cap total heat flux for
    # now
    total_heat_flux = jnp.clip(atm["derived"]["total_heat_flux"], min=-2000.0, max=2000.0)

    # Mapping
    ocn["forcing"].surface_taux = surface_taux
    ocn["forcing"].surface_tauy = surface_tauy     
    ocn["forcing"].heat_flux = jcm_to_veros_regridder(total_heat_flux)
    ocn["forcing"].freshwater_flux = jcm_to_veros_regridder(atm["derived"]["total_freshwater_flux"])
    ocn["forcing"].wind_x = jcm_to_veros_regridder(atm["derived"]["physics"].surface_flux.u0)
    ocn["forcing"].wind_y = jcm_to_veros_regridder(atm["derived"]["physics"].surface_flux.v0)
    slab_ocn["forcing"].total_heat_flux = jcm_to_veros_regridder(total_heat_flux)
    atm["forcing"].sea_surface_temperature = veros_to_jcm_regridder(ocn["derived"]["sea_surface_temperature"])
    atm["forcing"].stl_am = slab_ocn["state"]["sea_surface_temperature"]
    
    return coupled_carry

## Create Coupled Model

In [9]:
model = Coupler(
    components=dict(
        atm=atm_model,
        ocn=ocn_model,
        slab_ocn=slab_ocn_model,
    ),
    mappers=dict(mapper=interaction),
)

print("Model info: ") 
tree_tools.print_tree(model.get_info(), root="Model")

Checking `initialize` => `typing.Callable[[], typing.Any]`
Checking `generate_step_function` => `typing.Callable[[], typing.Callable[[typing.Any, float], tuple[typing.Any, typing.Any]]]`
Checking `predictions_to_xarray` => `typing.Callable[[typing.Any], xarray.core.dataset.Dataset]`
Checking `get_info` => `typing.Callable[[], typing.Dict]`
Checking `initialize` => `typing.Callable[[], typing.Any]`
Checking `generate_step_function` => `typing.Callable[[], typing.Callable[[typing.Any, float], tuple[typing.Any, typing.Any]]]`
Checking `predictions_to_xarray` => `typing.Callable[[typing.Any], xarray.core.dataset.Dataset]`
Checking `get_info` => `typing.Callable[[], typing.Dict]`
Checking `initialize` => `typing.Callable[[], typing.Any]`
Checking `generate_step_function` => `typing.Callable[[], typing.Callable[[typing.Any, float], tuple[typing.Any, typing.Any]]]`
Checking `predictions_to_xarray` => `typing.Callable[[typing.Any], xarray.core.dataset.Dataset]`
Checking `get_info` => `typing.C

## Run Coupled Model

In [10]:
initial_carry = model.initialize()
simulation_interval = jdt.to_timedelta(5, "day")
batches = int(total_simulation_time / simulation_interval)

for b in range(batches):
    
    print(f"[batch={b:d}/{batches:d}] Simulation...")
    
    _, final_carry, predictions = model.run(
        initial_carry = initial_carry,
        workflow=["mapper", "ocn", "atm", "slab_ocn"],
        iterations = int(simulation_interval / coupling_timestep),
        jitted=True,
        reuse_last_available_trajectory=True,
    )
    
    output_dict = model.predictions_to_xarray(predictions)

    ds_atm = output_dict["atm"]
    wind_mag = ((ds_atm["surface_flux.u0"]**2 + ds_atm["surface_flux.v0"]**2)**0.5).rename("wind_mag")

    output_dict["atm"] = xr.merge([
        ds_atm["specific_humidity"].isel(level=0),
        ds_atm["surface_flux.tsfc"],
        ds_atm["surface_flux.tskin"],
        ds_atm["shortwave_rad.rsns"],
        ds_atm["convection.precnv"],
        ds_atm["normalized_surface_pressure"],
        ds_atm["surface_flux.u0"],
        ds_atm["surface_flux.v0"],
        wind_mag,
    ])
    
    ocn = output_dict["ocn"]
    output_dict["ocn_daily"] = xr.merge([
        ocn["sea_surface_temperature"],
        ocn["sea_surface_u"],
        ocn["sea_surface_v"],
        ocn["heat_flux"],
    ])
    output_dict["ocn_mean"] = ocn.reduce(np.mean, dim="time", keepdims=True)
    del output_dict["ocn"]
    
    for component_name, ds in output_dict.items():
        output_file = output_dir / f"{component_name:s}-{b:03d}.nc"
        print("Output file: ", str(output_file))
        ds.to_netcdf(output_file, engine="netcdf4")
        ds.close()
   
    if jnp.any( jnp.isnan(output_dict["atm"]["specific_humidity"].to_numpy()) ):
        print("Error: Model exploded. End program")
        break

    initial_carry = final_carry

initial_forcing.surface_taux.shape = (96, 48)
Boundary does not exist. Idealized initial SST will be used.
grid.bmask and SST_clim do share the same mask.
Notice: Climaology SST does not exist. Set relaxation time to inifinity.
[batch=0/2] Simulation...
Flattened workflow:  mapper, ocn, atm, slab_ocn
Veros: settings.enable_tke is set true
The original set_forcing in the VerosSetup object is replaced by this empty set_forcing function. JEM-veros will set the forcing in the step_function.
The original set_forcing in the VerosSetup object is replaced by this empty set_forcing function. JEM-veros will set the forcing in the step_function.
The original set_forcing in the VerosSetup object is replaced by this empty set_forcing function. JEM-veros will set the forcing in the step_function.
The original set_forcing in the VerosSetup object is replaced by this empty set_forcing function. JEM-veros will set the forcing in the step_function.
The original set_forcing in the VerosSetup object is re

Simulation:   0%|          | 0/5 [00:00<?, ?it/s]

Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/atm-000.nc
Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/slab_ocn-000.nc
Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/ocn_daily-000.nc
Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/ocn_mean-000.nc
[batch=1/2] Simulation...
Reuse last available trajectory.


Simulation:   0%|          | 0/5 [00:00<?, ?it/s]

Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/atm-001.nc
Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/slab_ocn-001.nc
Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/ocn_daily-001.nc
Output file:  /home/xtt/projects/project_jax-esm/jax-esm_repo/docs/notebooks/02_experimental/02_experimental_JCM_Veros/output_T31/02-02_experimental_JCM_Veros/ocn_mean-001.nc
